In [1]:
import requests

url = "https://fever.ai/download/fever/shared_task_dev.jsonl"

response = requests.get(url)

print("Status:", response.status_code)
print("Size:", len(response.content), "bytes")

Status: 200
Size: 4349935 bytes


In [22]:
import sys
import os

project_root = os.path.abspath("..")

print("Project root:")
print(project_root)

print("\nsrc exists:")
print(os.path.exists(os.path.join(project_root, "src")))

sys.path.insert(0, project_root)

Project root:
/Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination

src exists:
True


In [23]:
import importlib
import src.data.preprocessing as preprocessing

importlib.reload(preprocessing)

print("convert_fever" in dir(preprocessing))

True


In [2]:
import os

raw_dir = "../data/raw/fever"
os.makedirs(raw_dir, exist_ok=True)

raw_path = os.path.join(
    raw_dir,
    "shared_task_dev.jsonl"
)

with open(raw_path, "wb") as f:
    f.write(response.content)

print("Saved:", raw_path)

Saved: ../data/raw/fever/shared_task_dev.jsonl


In [3]:
print(os.path.exists(raw_path))

True


In [4]:
import json

records = []

with open(raw_path, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print("Number of records:", len(records))

Number of records: 19998


In [5]:
print(json.dumps(records[0], indent=2))

{
  "id": 91198,
  "verifiable": "NOT VERIFIABLE",
  "label": "NOT ENOUGH INFO",
  "claim": "Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League.",
  "evidence": [
    [
      [
        108548,
        null,
        null,
        null
      ]
    ]
  ]
}


In [6]:
print("Keys:")
print(records[0].keys())

print("\nLabel:")
print(records[0]["label"])

print("\nClaim:")
print(records[0]["claim"])

print("\nEvidence:")
print(json.dumps(records[0]["evidence"], indent=2))

Keys:
dict_keys(['id', 'verifiable', 'label', 'claim', 'evidence'])

Label:
NOT ENOUGH INFO

Claim:
Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League.

Evidence:
[
  [
    [
      108548,
      null,
      null,
      null
    ]
  ]
]


In [7]:
from collections import Counter

label_counts = Counter(
    record["label"]
    for record in records
)

print(label_counts)

Counter({'NOT ENOUGH INFO': 6666, 'SUPPORTS': 6666, 'REFUTES': 6666})


In [8]:
verifiable_counts = Counter(
    record["verifiable"]
    for record in records
)

print(verifiable_counts)

Counter({'VERIFIABLE': 13332, 'NOT VERIFIABLE': 6666})


In [9]:
for label in sorted(set(record["label"] for record in records)):
    print(
        "\nLabel:",
        label
    )

    for record in records:
        if record["label"] == label:
            print(record)
            break


Label: NOT ENOUGH INFO
{'id': 91198, 'verifiable': 'NOT VERIFIABLE', 'label': 'NOT ENOUGH INFO', 'claim': 'Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League.', 'evidence': [[[108548, None, None, None]]]}

Label: REFUTES
{'id': 111897, 'verifiable': 'VERIFIABLE', 'label': 'REFUTES', 'claim': 'Telemundo is a English-language television network.', 'evidence': [[[131371, 146144, 'Telemundo', 0]], [[131371, 146148, 'Telemundo', 1]], [[131371, 146150, 'Telemundo', 4], [131371, 146150, 'Hispanic_and_Latino_Americans', 0]], [[131371, 146151, 'Telemundo', 5]]]}

Label: SUPPORTS
{'id': 137334, 'verifiable': 'VERIFIABLE', 'label': 'SUPPORTS', 'claim': 'Fox 2000 Pictures released the film Soul Food.', 'evidence': [[[289914, 283015, 'Soul_Food_-LRB-film-RRB-', 0]], [[291259, 284217, 'Soul_Food_-LRB-film-RRB-', 0]], [[293412, 285960, 'Soul_Food_-LRB-film-RRB-', 0]], [[337212, 322620, 'Soul_Food_-LRB-film-RRB-', 0]], [[337214, 322622, 'Soul_F

In [10]:
supports_record = next(
    r for r in records
    if r["label"] == "SUPPORTS"
)

print("ID:", supports_record["id"])
print("Claim:", supports_record["claim"])

print("\nEvidence:")
print(
    json.dumps(
        supports_record["evidence"],
        indent=2
    )
)

ID: 137334
Claim: Fox 2000 Pictures released the film Soul Food.

Evidence:
[
  [
    [
      289914,
      283015,
      "Soul_Food_-LRB-film-RRB-",
      0
    ]
  ],
  [
    [
      291259,
      284217,
      "Soul_Food_-LRB-film-RRB-",
      0
    ]
  ],
  [
    [
      293412,
      285960,
      "Soul_Food_-LRB-film-RRB-",
      0
    ]
  ],
  [
    [
      337212,
      322620,
      "Soul_Food_-LRB-film-RRB-",
      0
    ]
  ],
  [
    [
      337214,
      322622,
      "Soul_Food_-LRB-film-RRB-",
      0
    ]
  ]
]


In [11]:
refutes_record = next(
    r for r in records
    if r["label"] == "REFUTES"
)

print("ID:", refutes_record["id"])
print("Claim:", refutes_record["claim"])

print("\nEvidence:")
print(
    json.dumps(
        refutes_record["evidence"],
        indent=2
    )
)

ID: 111897
Claim: Telemundo is a English-language television network.

Evidence:
[
  [
    [
      131371,
      146144,
      "Telemundo",
      0
    ]
  ],
  [
    [
      131371,
      146148,
      "Telemundo",
      1
    ]
  ],
  [
    [
      131371,
      146150,
      "Telemundo",
      4
    ],
    [
      131371,
      146150,
      "Hispanic_and_Latino_Americans",
      0
    ]
  ],
  [
    [
      131371,
      146151,
      "Telemundo",
      5
    ]
  ]
]


In [12]:
from collections import Counter

print(
    Counter(
        record["label"]
        for record in records
    )
)

Counter({'NOT ENOUGH INFO': 6666, 'SUPPORTS': 6666, 'REFUTES': 6666})


In [13]:
print(
    Counter(
        record["verifiable"]
        for record in records
    )
)

Counter({'VERIFIABLE': 13332, 'NOT VERIFIABLE': 6666})


In [14]:
supports_record = next(
    r for r in records
    if r["label"] == "SUPPORTS"
)

print("ID:", supports_record["id"])
print("Claim:", supports_record["claim"])
print(
    json.dumps(
        supports_record["evidence"],
        indent=2
    )
)

ID: 137334
Claim: Fox 2000 Pictures released the film Soul Food.
[
  [
    [
      289914,
      283015,
      "Soul_Food_-LRB-film-RRB-",
      0
    ]
  ],
  [
    [
      291259,
      284217,
      "Soul_Food_-LRB-film-RRB-",
      0
    ]
  ],
  [
    [
      293412,
      285960,
      "Soul_Food_-LRB-film-RRB-",
      0
    ]
  ],
  [
    [
      337212,
      322620,
      "Soul_Food_-LRB-film-RRB-",
      0
    ]
  ],
  [
    [
      337214,
      322622,
      "Soul_Food_-LRB-film-RRB-",
      0
    ]
  ]
]


In [15]:
refutes_record = next(
    r for r in records
    if r["label"] == "REFUTES"
)

print("ID:", refutes_record["id"])
print("Claim:", refutes_record["claim"])
print(
    json.dumps(
        refutes_record["evidence"],
        indent=2
    )
)

ID: 111897
Claim: Telemundo is a English-language television network.
[
  [
    [
      131371,
      146144,
      "Telemundo",
      0
    ]
  ],
  [
    [
      131371,
      146148,
      "Telemundo",
      1
    ]
  ],
  [
    [
      131371,
      146150,
      "Telemundo",
      4
    ],
    [
      131371,
      146150,
      "Hispanic_and_Latino_Americans",
      0
    ]
  ],
  [
    [
      131371,
      146151,
      "Telemundo",
      5
    ]
  ]
]


In [16]:
def count_evidence(record):
    evidence = record.get("evidence", [])

    if not evidence:
        return 0

    count = 0

    for evidence_set in evidence:
        count += len(evidence_set)

    return count


evidence_counts = [
    count_evidence(record)
    for record in records
]

print("Records:", len(records))
print("Records with evidence:",
      sum(c > 0 for c in evidence_counts))

print("Records without evidence:",
      sum(c == 0 for c in evidence_counts))

Records: 19998
Records with evidence: 19998
Records without evidence: 0


In [17]:
for target_label in [
    "SUPPORTS",
    "REFUTES",
    "NOT ENOUGH INFO"
]:
    record = next(
        r for r in records
        if r["label"] == target_label
    )

    print("\n" + "=" * 70)
    print("LABEL:", target_label)
    print("ID:", record["id"])
    print("CLAIM:", record["claim"])
    print("VERIFIABLE:", record["verifiable"])
    print("EVIDENCE:")
    print(record["evidence"])


LABEL: SUPPORTS
ID: 137334
CLAIM: Fox 2000 Pictures released the film Soul Food.
VERIFIABLE: VERIFIABLE
EVIDENCE:
[[[289914, 283015, 'Soul_Food_-LRB-film-RRB-', 0]], [[291259, 284217, 'Soul_Food_-LRB-film-RRB-', 0]], [[293412, 285960, 'Soul_Food_-LRB-film-RRB-', 0]], [[337212, 322620, 'Soul_Food_-LRB-film-RRB-', 0]], [[337214, 322622, 'Soul_Food_-LRB-film-RRB-', 0]]]

LABEL: REFUTES
ID: 111897
CLAIM: Telemundo is a English-language television network.
VERIFIABLE: VERIFIABLE
EVIDENCE:
[[[131371, 146144, 'Telemundo', 0]], [[131371, 146148, 'Telemundo', 1]], [[131371, 146150, 'Telemundo', 4], [131371, 146150, 'Hispanic_and_Latino_Americans', 0]], [[131371, 146151, 'Telemundo', 5]]]

LABEL: NOT ENOUGH INFO
ID: 91198
CLAIM: Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League.
VERIFIABLE: NOT VERIFIABLE
EVIDENCE:
[[[108548, None, None, None]]]


In [26]:
convert_fever = preprocessing.convert_fever

In [ ]:
import os

raw_dir = "../data/raw/fever"
os.makedirs(raw_dir, exist_ok=True)

raw_path = os.path.join(
    raw_dir,
    "shared_task_dev.jsonl"
)

with open(raw_path, "wb") as f:
    f.write(response.content)

print("Saved:", raw_path)

Saved: ../data/raw/fever/shared_task_dev.jsonl


In [ ]:
fever_binary = fever_df[
    fever_df["label"].isin(["SUPPORTS", "REFUTES"])
].copy()

print("Shape:", fever_binary.shape)
print("\nLabels:")
print(fever_binary["label"].value_counts())

Shape: (13332, 5)

Labels:
label
SUPPORTS    6666
REFUTES     6666
Name: count, dtype: int64


In [ ]:
fever_binary["binary_label"] = (
    fever_binary["label"]
    .map({
        "SUPPORTS": 0,
        "REFUTES": 1
    })
)

In [ ]:
print(fever_binary["binary_label"].value_counts())

binary_label
0    6666
1    6666
Name: count, dtype: int64


In [31]:
import pandas as pd

# 1. Create DataFrame from the raw FEVER records
fever_df = pd.DataFrame([
    {
        "id": record["id"],
        "verifiable": record["verifiable"],
        "label": record["label"],
        "claim": record["claim"],
        "evidence": record["evidence"]
    }
    for record in records
])

# 2. Keep only SUPPORTS and REFUTES
fever_binary = fever_df[
    fever_df["label"].isin(["SUPPORTS", "REFUTES"])
].copy()

# 3. Convert FEVER labels to our binary labels
fever_binary["binary_label"] = fever_binary["label"].map({
    "SUPPORTS": 0,
    "REFUTES": 1
})

# 4. Print everything so we know it worked
print("FEVER raw shape:", fever_df.shape)
print("FEVER binary shape:", fever_binary.shape)

print("\nOriginal labels:")
print(fever_binary["label"].value_counts())

print("\nBinary labels:")
print(fever_binary["binary_label"].value_counts())

FEVER raw shape: (19998, 5)
FEVER binary shape: (13332, 6)

Original labels:
label
SUPPORTS    6666
REFUTES     6666
Name: count, dtype: int64

Binary labels:
binary_label
0    6666
1    6666
Name: count, dtype: int64


In [32]:
import json

raw_path = "../data/raw/fever/shared_task_dev.jsonl"

records = []

with open(raw_path, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print("Records loaded:", len(records))


Records loaded: 19998


In [33]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

import importlib
import src.data.preprocessing as preprocessing

importlib.reload(preprocessing)

print("convert_fever exists:",
      hasattr(preprocessing, "convert_fever"))

convert_fever exists: True


In [34]:
convert_fever = preprocessing.convert_fever

fever_processed = convert_fever(
    fever_binary
)

print("Shape:", fever_processed.shape)
print("Columns:", fever_processed.columns.tolist())

Shape: (13332, 8)
Columns: ['question_id', 'candidate_id', 'source_dataset', 'question', 'answer', 'context', 'label', 'question_category']


In [35]:
print(
    fever_processed.head().to_string(index=False)
)

 question_id       candidate_id source_dataset                                            question                                              answer context  label  question_category
FEVER_137334 FEVER_137334_C_000          fever      Fox 2000 Pictures released the film Soul Food.      Fox 2000 Pictures released the film Soul Food.              0 claim_verification
FEVER_111897 FEVER_111897_C_000          fever Telemundo is a English-language television network. Telemundo is a English-language television network.              1 claim_verification
 FEVER_89891  FEVER_89891_C_000          fever    Damon Albarn's debut album was released in 2011.    Damon Albarn's debut album was released in 2011.              1 claim_verification
FEVER_181634 FEVER_181634_C_000          fever                There is a capital called Mogadishu.                There is a capital called Mogadishu.              0 claim_verification
FEVER_219028 FEVER_219028_C_000          fever              Savages was exc

In [36]:
print("\nLabels:")
print(fever_processed["label"].value_counts())


Labels:
label
0    6666
1    6666
Name: count, dtype: int64


In [37]:
print("\nUnique question IDs:",
      fever_processed["question_id"].nunique())

print("Unique candidate IDs:",
      fever_processed["candidate_id"].nunique())


Unique question IDs: 13332
Unique candidate IDs: 13332


In [38]:
print("Missing values:")
print(fever_processed.isnull().sum())

print(
    "\nEmpty questions:",
    fever_processed["question"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Empty answers:",
    fever_processed["answer"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Duplicate Q&A pairs:",
    fever_processed.duplicated(
        subset=["question", "answer"]
    ).sum()
)

Missing values:
question_id          0
candidate_id         0
source_dataset       0
question             0
answer               0
context              0
label                0
question_category    0
dtype: int64

Empty questions: 0
Empty answers: 0
Duplicate Q&A pairs: 243


In [39]:
duplicates = fever_processed[
    fever_processed.duplicated(
        subset=["question", "answer"],
        keep=False
    )
].sort_values(["question", "answer"])

print("Rows involved in duplicate pairs:", len(duplicates))

print(
    duplicates[
        [
            "question_id",
            "candidate_id",
            "question",
            "answer",
            "label"
        ]
    ].head(30).to_string(index=False)
)

Rows involved in duplicate pairs: 448
 question_id       candidate_id                                               question                                                 answer  label
FEVER_116668 FEVER_116668_C_000                        A Milli is a song by Lil Wayne.                        A Milli is a song by Lil Wayne.      0
 FEVER_21236  FEVER_21236_C_000                        A Milli is a song by Lil Wayne.                        A Milli is a song by Lil Wayne.      0
FEVER_192790 FEVER_192790_C_000                          A staging area is a location.                          A staging area is a location.      0
FEVER_192775 FEVER_192775_C_000                          A staging area is a location.                          A staging area is a location.      0
FEVER_192795 FEVER_192795_C_000        A staging area is only an unused piece of land.        A staging area is only an unused piece of land.      1
FEVER_192792 FEVER_192792_C_000        A staging area is only an unu

In [40]:
conflict_check = (
    fever_processed
    .groupby(["question", "answer"])["label"]
    .nunique()
)

conflicting_pairs = conflict_check[
    conflict_check > 1
]

print(
    "Conflicting Q&A pairs:",
    len(conflicting_pairs)
)

Conflicting Q&A pairs: 3


In [41]:
duplicate_label_counts = (
    duplicates
    .groupby(["question", "answer"])["label"]
    .nunique()
)

print(
    duplicate_label_counts.value_counts()
)

label
1    202
2      3
Name: count, dtype: int64


In [42]:
conflicting_pairs = (
    fever_processed
    .groupby(["question", "answer"])["label"]
    .nunique()
)

conflicting_pairs = conflicting_pairs[
    conflicting_pairs > 1
]

print("Number of conflicting pairs:", len(conflicting_pairs))

for question, answer in conflicting_pairs.index:
    print("\n" + "=" * 70)
    print("QUESTION:", question)
    print("ANSWER:", answer)

    print(
        fever_processed[
            (fever_processed["question"] == question) &
            (fever_processed["answer"] == answer)
        ][
            [
                "question_id",
                "label"
            ]
        ].to_string(index=False)
    )

Number of conflicting pairs: 3

QUESTION: An island is part of the ABC Islands.
ANSWER: An island is part of the ABC Islands.
 question_id  label
FEVER_139910      0
 FEVER_26450      1

QUESTION: Janet Leigh was a person.
ANSWER: Janet Leigh was a person.
 question_id  label
FEVER_146011      0
 FEVER_37792      0
FEVER_129419      1

QUESTION: The Hundred Years' War includes the Civil War.
ANSWER: The Hundred Years' War includes the Civil War.
question_id  label
FEVER_54077      1
  FEVER_519      0


In [43]:
conflict_index = pd.MultiIndex.from_frame(
    fever_processed[
        ["question", "answer"]
    ]
)

fever_no_conflicts = fever_processed[
    ~conflict_index.isin(
        conflicting_pairs.index
    )
].copy()

In [44]:
print(
    "Before removing conflicts:",
    len(fever_processed)
)

print(
    "After removing conflicts:",
    len(fever_no_conflicts)
)

Before removing conflicts: 13332
After removing conflicts: 13325


In [45]:
fever_cleaned = fever_no_conflicts.drop_duplicates(
    subset=["question", "answer"],
    keep="first"
).reset_index(drop=True)

In [46]:
print("Before deduplication:", len(fever_no_conflicts))
print("After deduplication:", len(fever_cleaned))

print(
    "\nDuplicate Q&A pairs remaining:",
    fever_cleaned.duplicated(
        subset=["question", "answer"]
    ).sum()
)

Before deduplication: 13325
After deduplication: 13086

Duplicate Q&A pairs remaining: 0


In [47]:
print(
    fever_cleaned["label"].value_counts()
)

label
0    6549
1    6537
Name: count, dtype: int64


In [48]:
print("Final FEVER shape:", fever_cleaned.shape)

print(
    "\nUnique questions:",
    fever_cleaned["question_id"].nunique()
)

print(
    "Unique candidates:",
    fever_cleaned["candidate_id"].nunique()
)

print(
    "\nMissing values:"
)

print(
    fever_cleaned.isnull().sum()
)

print(
    "\nDuplicate Q&A pairs:",
    fever_cleaned.duplicated(
        subset=["question", "answer"]
    ).sum()
)

Final FEVER shape: (13086, 8)

Unique questions: 13086
Unique candidates: 13086

Missing values:
question_id          0
candidate_id         0
source_dataset       0
question             0
answer               0
context              0
label                0
question_category    0
dtype: int64

Duplicate Q&A pairs: 0


In [49]:
conflict_check = (
    fever_cleaned
    .groupby(["question", "answer"])["label"]
    .nunique()
)

print(
    "\nRemaining conflicting pairs:",
    (conflict_check > 1).sum()
)


Remaining conflicting pairs: 0


In [50]:
from sklearn.model_selection import train_test_split
import numpy as np

question_ids = np.array(
    fever_cleaned["question_id"].unique()
)

print("Total claims:", len(question_ids))

Total claims: 13086


In [51]:
train_ids, temp_ids = train_test_split(
    question_ids,
    test_size=0.30,
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42
)

print("Train claims:", len(train_ids))
print("Validation claims:", len(val_ids))
print("Test claims:", len(test_ids))

Train claims: 9160
Validation claims: 1963
Test claims: 1963


In [52]:
fever_train = fever_cleaned[
    fever_cleaned["question_id"].isin(train_ids)
].copy()

fever_val = fever_cleaned[
    fever_cleaned["question_id"].isin(val_ids)
].copy()

fever_test = fever_cleaned[
    fever_cleaned["question_id"].isin(test_ids)
].copy()

In [53]:
print("Train rows:", len(fever_train))
print("Validation rows:", len(fever_val))
print("Test rows:", len(fever_test))

Train rows: 9160
Validation rows: 1963
Test rows: 1963


In [54]:
train_questions = set(
    fever_train["question_id"]
)

val_questions = set(
    fever_val["question_id"]
)

test_questions = set(
    fever_test["question_id"]
)

print(
    "Train ∩ Validation:",
    len(train_questions & val_questions)
)

print(
    "Train ∩ Test:",
    len(train_questions & test_questions)
)

print(
    "Validation ∩ Test:",
    len(val_questions & test_questions)
)

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [55]:
fever_train.to_parquet(
    "../data/processed/fever_train.parquet",
    index=False
)

fever_val.to_parquet(
    "../data/processed/fever_validation.parquet",
    index=False
)

fever_test.to_parquet(
    "../data/processed/fever_test.parquet",
    index=False
)

print("FEVER splits saved successfully!")

FEVER splits saved successfully!
